# Transformer-based Architecture Module — TrOCR for Handwritten Prescription OCR

Pipeline: **setup → data → model → training (resumable) → evaluation → export (`.safetensors` → `.pth` for GUI use)**

Run the cells top to bottom. Every cell is written to be **safe to re-run** (no duplicate extraction, no overwriting a better checkpoint, no hard-coded GPU asserts).

### What changed in this version (fixes for slow / disconnecting runs)
* **Generation fix** – `eos/pad/decoder_start` are now set on `generation_config`, and `max_new_tokens` is 24 (labels are ≤10 tokens). Before, some predictions ran to 128 tokens, which gave CER > 1 and made test evaluation extremely slow (it disconnected).
* **Images are copied from Drive to local disk** once, so training no longer reads thousands of files over the Drive mount.
* **Only the latest `optimizer.pt` is kept** (it is ~2.7 GB per epoch), so Drive no longer fills up.
* `TOTAL_EPOCHS` = 5 (validation loss stops improving after ~epoch 2–6, so more epochs only overfit).
* The exported GUI loader sets the same generation settings.

**Already trained?** Run sections 0–1, then **skip sections 2–3** (cells that load the base model, build loaders and train) and go straight to **Evaluation** (it loads `best_trocr_model` from Drive).


## 0. Setup

In [ ]:
# GPU check
!nvidia-smi

import torch
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU. Runtime > Change runtime type > T4 GPU (training on CPU is extremely slow).")

In [ ]:
# Install only what is missing (safe to re-run; no version pins, so no dependency conflicts)
import importlib, subprocess, sys

def ensure(pip_name, module_name=None):
    module_name = module_name or pip_name
    try:
        importlib.import_module(module_name)
        print(f"✅ {pip_name} already installed")
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        importlib.invalidate_caches()
        importlib.import_module(module_name)
        print(f"✅ {pip_name} installed")

ensure("sentencepiece")
ensure("jiwer")

In [ ]:
# Verify the environment (auto-installs anything still missing, then re-checks)
import importlib, subprocess, sys

PACKAGES = {  # import name -> pip name
    "torch": "torch", "transformers": "transformers", "tokenizers": "tokenizers",
    "sentencepiece": "sentencepiece", "jiwer": "jiwer", "PIL": "pillow", "sklearn": "scikit-learn",
}

for mod, pip_name in PACKAGES.items():
    try:
        m = importlib.import_module(mod)
    except ImportError:
        print(f"⚠️ {mod} missing - installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        importlib.invalidate_caches()
        m = importlib.import_module(mod)
    print(f"✅ {mod:14s}", getattr(m, "__version__", ""))

from transformers import TrOCRProcessor, VisionEncoderDecoderModel
_p = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")   # tests tokenizer + sentencepiece
print("✅ TrOCR processor loads correctly")
del _p

In [ ]:
import os, re, json, random, shutil, zipfile, textwrap, unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split

import jiwer   # NOTE: use jiwer.wer / jiwer.cer — never `from jiwer import wer` (it gets shadowed by variables)
import transformers
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, VisionEncoderDecoderConfig

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Transformers:", transformers.__version__)
print("Device      :", device)

### Mount Google Drive and define all paths/settings in one place

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ----------------------------- PATHS -----------------------------
PROJECT_ROOT   = "/content/drive/MyDrive/GenAI_Transformer_Model"
ZIP_PATH       = f"{PROJECT_ROOT}/RxHandBD-ML.zip"          # dataset zip (file, not a folder)
PROJECT_DIR    = f"{PROJECT_ROOT}/RxHandBD-ML"
CHECKPOINT_DIR = f"{PROJECT_DIR}/TrOCR_Checkpoints"
SPLIT_DIR      = f"{CHECKPOINT_DIR}/dataset_splits"
EVAL_DIR       = f"{CHECKPOINT_DIR}/evaluation"
EXTRACT_DIR    = "/content/RxHandBD-ML"                       # fast local copy of the images

# ----------------------------- MODEL / TRAINING SETTINGS -----------------------------
MODEL_NAME        = "microsoft/trocr-base-handwritten"
MAX_TARGET_LENGTH = 128
BATCH_SIZE        = 4          # lower to 2 if you get CUDA out-of-memory
TOTAL_EPOCHS      = 5          # train until this epoch number (val loss stops improving early)
START_EPOCH       = 0          # 0 = fresh run from MODEL_NAME; 5 = resume from trocr_epoch5, etc.
LEARNING_RATE     = 5e-5 if START_EPOCH == 0 else 1e-5

# Which saved model to evaluate / export ("best_trocr_model" always exists after training;
# use e.g. "trocr_epoch8" to pick a specific epoch)
FINAL_MODEL_NAME = "best_trocr_model"

for d in (CHECKPOINT_DIR, SPLIT_DIR, EVAL_DIR):
    os.makedirs(d, exist_ok=True)

print("Dataset zip exists :", os.path.isfile(ZIP_PATH))
print("Checkpoint dir     :", CHECKPOINT_DIR)

## 1. Data

In [ ]:
# ---- DATASET LOCATOR + EXTRACTOR (replaces the previous extraction cell) ----
MANUAL_ZIP_PATH = ""     # optional: paste the exact path of your dataset zip here, e.g. "/content/drive/MyDrive/xyz/data.zip"
ALLOW_UPLOAD    = True   # if nothing is found in Drive, ask you to upload the zip from your computer

DRIVE_ROOT = "/content/drive/MyDrive"
assert os.path.isdir(DRIVE_ROOT), "Google Drive is not mounted - run the 'Mount Google Drive' cell first."

def show_tree(path, max_depth=3, depth=0, limit=30):
    try:
        names = sorted(os.listdir(path))
    except Exception:
        return
    for n in names[:limit]:
        full = os.path.join(path, n)
        print("    " * depth + ("📁 " if os.path.isdir(full) else "📄 ") + n)
        if os.path.isdir(full) and depth + 1 < max_depth:
            show_tree(full, max_depth, depth + 1, limit)
    if len(names) > limit:
        print("    " * depth + f"... (+{len(names) - limit} more)")

def search_drive():
    csv_dirs, zips, set_dirs = [], [], []
    for root, dirs, files in os.walk(DRIVE_ROOT):
        dirs[:] = [d for d in dirs if d != ".Trash"]
        for d in dirs:
            if d.lower() == "train_set":
                set_dirs.append(os.path.join(root, d))
        for f in files:
            low = f.lower()
            if low == "train_label.csv":
                csv_dirs.append(root)
            elif low.endswith(".zip"):
                full = os.path.join(root, f)
                try:
                    zips.append((full, os.path.getsize(full) / 1024**2))
                except OSError:
                    pass
    return csv_dirs, zips, set_dirs

dataset_zip, dataset_folder = None, None

if MANUAL_ZIP_PATH:
    assert os.path.isfile(MANUAL_ZIP_PATH), f"MANUAL_ZIP_PATH does not exist: {MANUAL_ZIP_PATH}"
    dataset_zip = MANUAL_ZIP_PATH
elif os.path.isfile(ZIP_PATH):
    dataset_zip = ZIP_PATH
else:
    print("Searching Google Drive for the dataset ...")
    csv_dirs, zips, set_dirs = search_drive()
    if csv_dirs:
        dataset_folder = csv_dirs[0]
    elif set_dirs:
        dataset_folder = os.path.dirname(set_dirs[0])
    else:
        likely = [z for z, s in zips if any(k in os.path.basename(z).lower()
                                            for k in ("rxhand", "handbd", "prescription"))]
        if likely:
            dataset_zip = likely[0]

if dataset_zip is None and dataset_folder is None:
    print("\n❌ The dataset is NOT in this Google Drive account.\n")
    print("What your project folder actually contains:")
    show_tree(PROJECT_ROOT)
    if ALLOW_UPLOAD:
        from google.colab import files
        print("\nChoose the dataset ZIP from your computer (it will be copied to Drive for next time):")
        uploaded = files.upload()
        names = [n for n in uploaded if n.lower().endswith(".zip")]
        if not names:
            raise FileNotFoundError("No .zip was uploaded.")
        dataset_zip = os.path.join(PROJECT_ROOT, "RxHandBD-ML.zip")
        os.makedirs(PROJECT_ROOT, exist_ok=True)
        shutil.copy(os.path.join("/content", names[0]), dataset_zip)
        print("✅ Saved to Drive:", dataset_zip)
    else:
        raise FileNotFoundError("Put the dataset zip in Google Drive or set MANUAL_ZIP_PATH.")

if dataset_zip:
    ZIP_PATH = dataset_zip
    print("Using dataset zip:", ZIP_PATH)
    if os.path.isdir(EXTRACT_DIR) and any(os.scandir(EXTRACT_DIR)):
        print("Dataset already extracted:", EXTRACT_DIR)
    else:
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(EXTRACT_DIR)
        print("✅ Dataset extracted to", EXTRACT_DIR)
else:
    LOCAL_COPY = "/content/RxHandBD-ML"
    if os.path.isdir(LOCAL_COPY) and any(os.scandir(LOCAL_COPY)):
        print("Local copy already exists:", LOCAL_COPY)
    else:
        print("Copying dataset from Drive to local disk (one-time; makes training much faster) ...")
        shutil.copytree(dataset_folder, LOCAL_COPY, dirs_exist_ok=True)
    EXTRACT_DIR = LOCAL_COPY
    print("Using LOCAL dataset copy:", EXTRACT_DIR)


In [ ]:
# Locate label CSVs and image folders automatically
TRAIN_CSV = TEST_CSV = TRAIN_IMG_DIR = TEST_IMG_DIR = None

for root, dirs, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.lower() == "train_label.csv":
            TRAIN_CSV = os.path.join(root, f)
        elif f.lower() == "test_label.csv":
            TEST_CSV = os.path.join(root, f)
    for d in dirs:
        if d.lower() == "train_set":
            TRAIN_IMG_DIR = os.path.join(root, d)
        elif d.lower() == "test_set":
            TEST_IMG_DIR = os.path.join(root, d)

print("TRAIN CSV    :", TRAIN_CSV)
print("TEST CSV     :", TEST_CSV)
print("TRAIN IMAGES :", TRAIN_IMG_DIR)
print("TEST IMAGES  :", TEST_IMG_DIR)

assert TRAIN_CSV and TEST_CSV, "Label CSV not found"
assert TRAIN_IMG_DIR and TEST_IMG_DIR, "Train_Set / Test_Set folder not found"

In [ ]:
def standardize(df):
    """Make sure the frame has exactly the columns 'Images' and 'Text'."""
    if not {"Images", "Text"}.issubset(df.columns):
        print("⚠️ Expected columns not found, using first two columns:", df.columns[:2].tolist())
        df = df.rename(columns={df.columns[0]: "Images", df.columns[1]: "Text"})
    return df[["Images", "Text"]].copy()

train_df = standardize(pd.read_csv(TRAIN_CSV))
test_df  = standardize(pd.read_csv(TEST_CSV))

print("Train:", train_df.shape, "| Test:", test_df.shape)
print("\nMissing values (train):\n", train_df.isnull().sum())
print("\nMissing values (test):\n", test_df.isnull().sum())
print("\nDuplicate image ids  train/test:",
      train_df["Images"].duplicated().sum(), "/", test_df["Images"].duplicated().sum())
display(train_df.head())

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_frame(df):
    df = df.dropna(how="all").copy()
    df["Images"] = df["Images"].astype(str).str.strip()
    df["Text"]   = df["Text"].apply(clean_text)
    df = df[(df["Images"] != "") & (df["Text"] != "")]
    df = df.drop_duplicates(subset=["Images"])
    return df.reset_index(drop=True)

train_df = clean_frame(train_df)
test_df  = clean_frame(test_df)
print("Clean train:", len(train_df), "| Clean test:", len(test_df))

In [ ]:
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".webp")

def build_lookup(img_dir):
    """filename (with and without extension) -> full path, per split folder.
    (The old notebook used ONE lookup for everything, so identical file names in
    Train_Set and Test_Set could silently point to the wrong image.)"""
    lookup = {}
    for root, _, files in os.walk(img_dir):
        for f in files:
            if f.lower().endswith(IMAGE_EXTENSIONS):
                p = os.path.join(root, f)
                lookup[f] = p
                lookup.setdefault(os.path.splitext(f)[0], p)
    return lookup

def resolve(lookup, image_id):
    image_id = os.path.basename(str(image_id).strip())
    return lookup.get(image_id) or lookup.get(os.path.splitext(image_id)[0])

train_lookup = build_lookup(TRAIN_IMG_DIR)
test_lookup  = build_lookup(TEST_IMG_DIR)

train_df["image_path"] = train_df["Images"].apply(lambda x: resolve(train_lookup, x))
test_df["image_path"]  = test_df["Images"].apply(lambda x: resolve(test_lookup, x))

print("Unmatched train images:", train_df["image_path"].isna().sum())
print("Unmatched test images :", test_df["image_path"].isna().sum())

train_df = train_df.dropna(subset=["image_path"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["image_path"]).reset_index(drop=True)
print("Final train:", len(train_df), "| Final test:", len(test_df))

In [ ]:
# Look at a few samples
plt.figure(figsize=(15, 8))
for i in range(min(6, len(train_df))):
    row = train_df.iloc[i]
    plt.subplot(2, 3, i + 1)
    plt.imshow(Image.open(row["image_path"]).convert("RGB"))
    plt.title(textwrap.fill(row["Text"], 35), fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

# Image-size and text-length statistics (once, on a sample)
sizes = []
for p in train_df["image_path"].head(1000):
    try:
        with Image.open(p) as im:
            sizes.append(im.size)
    except Exception:
        pass
if sizes:
    w, h = zip(*sizes)
    print(f"Checked {len(sizes)} images | width {min(w)}-{max(w)} | height {min(h)}-{max(h)}")
print("\nText length (characters):")
print(train_df["Text"].str.len().describe())

### Train / validation / test split — saved to Drive

The old notebook later read `dataset_splits/train.csv`, but **no cell ever created it**, and it re-split the data in two more places (risking train/validation leakage after a resume).
Now the split is created **once**, saved, and simply reloaded in later sessions.

In [ ]:
split_files = {n: os.path.join(SPLIT_DIR, f"{n}.csv") for n in ("train", "val", "test")}

if all(os.path.exists(p) for p in split_files.values()):
    print("Loading existing splits from Drive (keeps train/val separation identical across sessions)")
    train_data = pd.read_csv(split_files["train"])
    val_data   = pd.read_csv(split_files["val"])
    test_data  = pd.read_csv(split_files["test"])
else:
    print("Creating new splits and saving them to Drive")
    train_data, val_data = train_test_split(train_df, test_size=0.10, random_state=SEED, shuffle=True)
    train_data = train_data.reset_index(drop=True)
    val_data   = val_data.reset_index(drop=True)
    test_data  = test_df.copy()
    train_data.to_csv(split_files["train"], index=False)
    val_data.to_csv(split_files["val"],     index=False)
    test_data.to_csv(split_files["test"],   index=False)

# Re-point image paths to THIS runtime's extraction folder (paths change between Colab sessions)
if "train_lookup" not in globals():
    train_lookup = build_lookup(TRAIN_IMG_DIR)
    test_lookup  = build_lookup(TEST_IMG_DIR)
for df_, lk in ((train_data, train_lookup), (val_data, train_lookup), (test_data, test_lookup)):
    df_["image_path"] = df_["Images"].apply(lambda x: resolve(lk, x))
train_data = train_data.dropna(subset=["image_path"]).reset_index(drop=True)
val_data   = val_data.dropna(subset=["image_path"]).reset_index(drop=True)
test_data  = test_data.dropna(subset=["image_path"]).reset_index(drop=True)

assert train_data["Images"].isin(val_data["Images"]).sum() == 0, "Train/val overlap!"
print(f"Train: {len(train_data)} | Validation: {len(val_data)} | Test: {len(test_data)}")

## 2. Model, dataset and data loaders

In [ ]:
# Fresh start from the pretrained TrOCR, or resume from a saved epoch checkpoint
INIT_FROM = None if START_EPOCH == 0 else os.path.join(CHECKPOINT_DIR, f"trocr_epoch{START_EPOCH}")
SOURCE = INIT_FROM or MODEL_NAME
if INIT_FROM and not os.path.isdir(INIT_FROM):
    raise FileNotFoundError(f"Cannot resume, checkpoint not found: {INIT_FROM}")

print("Loading from:", SOURCE)
processor = TrOCRProcessor.from_pretrained(SOURCE)
model     = VisionEncoderDecoderModel.from_pretrained(SOURCE)

# Required TrOCR generation/training config
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.sep_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# Generation settings (transformers >= 5 reads these from generation_config) - stops runaway outputs
model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.generation_config.pad_token_id           = processor.tokenizer.pad_token_id
model.generation_config.eos_token_id           = processor.tokenizer.sep_token_id

model = model.to(device)

print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("pad_token_id          :", model.config.pad_token_id)
print("eos_token_id          :", model.config.eos_token_id)
print("Model device          :", next(model.parameters()).device)

In [ ]:
class PrescriptionDataset(Dataset):
    """One dataset class used everywhere (the old notebook defined it 3 times with different signatures)."""
    def __init__(self, dataframe, processor, image_col="image_path", text_col="Text",
                 max_target_length=128):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
        self.image_col = image_col
        self.text_col = text_col
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row[self.image_col]).convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            str(row[self.text_col]),
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100   # ignore padding in loss

        return {"pixel_values": pixel_values, "labels": labels}

train_dataset = PrescriptionDataset(train_data, processor, max_target_length=MAX_TARGET_LENGTH)
val_dataset   = PrescriptionDataset(val_data,   processor, max_target_length=MAX_TARGET_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=torch.cuda.is_available())

print("Train batches:", len(train_loader), "| Validation batches:", len(val_loader))

# Sanity check: one sample + label truncation check
s = train_dataset[0]
print("pixel_values:", tuple(s["pixel_values"].shape), s["pixel_values"].dtype,
      "| labels:", tuple(s["labels"].shape), s["labels"].dtype)

lens = [len(processor.tokenizer(t).input_ids) for t in train_data["Text"].head(2000)]
too_long = sum(l > MAX_TARGET_LENGTH for l in lens)
print(f"Labels longer than {MAX_TARGET_LENGTH} tokens (would be truncated): {too_long}/{len(lens)}  (max {max(lens)})")

## 3. Training (fresh or resumable)

* Mixed precision (AMP) + gradient clipping.
* Saves **every epoch** and the **best** model directly to Google Drive.
* On resume, `best_val_loss` is restored from the saved log, so a worse epoch can no longer overwrite `best_trocr_model` (this was a bug in the old continue-training cell).
* Set `START_EPOCH` / `TOTAL_EPOCHS` in the settings cell.

In [ ]:
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def run_epoch(model, loader, device, optimizer=None, scaler=None, desc="", max_grad_norm=1.0):
    """Train if optimizer is given, otherwise validate. Returns mean loss."""
    training = optimizer is not None
    model.train() if training else model.eval()
    total = 0.0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        bar = tqdm(loader, desc=desc)
        for batch in bar:
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            labels       = batch["labels"].to(device, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = model(pixel_values=pixel_values, labels=labels).loss

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()

            total += loss.item()
            bar.set_postfix(loss=f"{loss.item():.4f}")

    return total / len(loader)

In [ ]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Resume optimizer state if it was saved with the epoch checkpoint
if INIT_FROM:
    opt_path = os.path.join(INIT_FROM, "optimizer.pt")
    if os.path.exists(opt_path):
        optimizer.load_state_dict(torch.load(opt_path, map_location=device))
        for g in optimizer.param_groups:
            g["lr"] = LEARNING_RATE
        print("✅ Optimizer state restored from", opt_path)
    else:
        print("ℹ️ No optimizer.pt found - starting with a fresh optimizer")

# Training log (list of {epoch, train_loss, val_loss}), restored on resume
LOG_PATH = os.path.join(CHECKPOINT_DIR, "training_log.json")
history = []
if START_EPOCH > 0 and os.path.exists(LOG_PATH):
    history = [h for h in json.load(open(LOG_PATH)) if h["epoch"] <= START_EPOCH]
best_val_loss = min((h["val_loss"] for h in history), default=float("inf"))

print(f"Epochs {START_EPOCH + 1} -> {TOTAL_EPOCHS} | lr={LEARNING_RATE} | best val loss so far: {best_val_loss}")

for epoch in range(START_EPOCH, TOTAL_EPOCHS):
    n = epoch + 1
    print("\n" + "=" * 60 + f"\nEPOCH {n}/{TOTAL_EPOCHS}\n" + "=" * 60)

    train_loss = run_epoch(model, train_loader, device, optimizer, scaler, desc=f"Epoch {n} train")
    val_loss   = run_epoch(model, val_loader,   device,            desc=f"Epoch {n} val")

    history.append({"epoch": n, "train_loss": train_loss, "val_loss": val_loss})
    print(f"Train loss: {train_loss:.4f} | Validation loss: {val_loss:.4f}")

    # Save every epoch to Drive
    epoch_dir = os.path.join(CHECKPOINT_DIR, f"trocr_epoch{n}")
    os.makedirs(epoch_dir, exist_ok=True)
    model.save_pretrained(epoch_dir)
    processor.save_pretrained(epoch_dir)
    torch.save(optimizer.state_dict(), os.path.join(epoch_dir, "optimizer.pt"))
    # optimizer.pt is ~2.7 GB: keep only the newest one so Drive does not fill up
    prev_opt = os.path.join(CHECKPOINT_DIR, f"trocr_epoch{n-1}", "optimizer.pt")
    if os.path.exists(prev_opt):
        os.remove(prev_opt)
    print(f"✅ Epoch {n} saved -> {epoch_dir}")

    # Save best model (only if it really beats every previous epoch, including before a resume)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_dir = os.path.join(CHECKPOINT_DIR, "best_trocr_model")
        os.makedirs(best_dir, exist_ok=True)
        model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)
        with open(os.path.join(best_dir, "best_info.json"), "w") as f:
            json.dump({"epoch": n, "val_loss": val_loss}, f)
        print(f"🏆 New best model (epoch {n}, val loss {val_loss:.4f})")

    json.dump(history, open(LOG_PATH, "w"), indent=2)
    torch.cuda.empty_cache()

print("\n🎉 TRAINING COMPLETED | best validation loss:", best_val_loss)

In [ ]:
# Loss curves (read from the saved log, so the plot also works after a resume / in a new session)
log = pd.DataFrame(json.load(open(LOG_PATH)))
display(log)

plt.figure(figsize=(10, 5))
plt.plot(log["epoch"], log["train_loss"], marker="o", label="Training loss")
plt.plot(log["epoch"], log["val_loss"],   marker="o", label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("TrOCR training vs validation loss")
plt.legend(); plt.grid(True); plt.show()

## 4. Evaluation

Metrics are computed on the **held-out validation and test sets**.
(The old final cell evaluated on the *training* CSV, which only measures memorisation, and then plotted every single image.)

In [ ]:
FINAL_MODEL_DIR = os.path.join(CHECKPOINT_DIR, FINAL_MODEL_NAME)
if not os.path.isdir(FINAL_MODEL_DIR):
    raise FileNotFoundError(f"Model folder not found: {FINAL_MODEL_DIR}\n"
                            f"Available: {sorted(os.listdir(CHECKPOINT_DIR))}")

eval_processor = TrOCRProcessor.from_pretrained(FINAL_MODEL_DIR, local_files_only=True)
eval_model     = VisionEncoderDecoderModel.from_pretrained(FINAL_MODEL_DIR, local_files_only=True).to(device)
eval_model.eval()

# Make sure generation stops at end-of-sequence (older checkpoints do not have this saved)
_tok = eval_processor.tokenizer
eval_model.generation_config.decoder_start_token_id = _tok.cls_token_id
eval_model.generation_config.pad_token_id           = _tok.pad_token_id
eval_model.generation_config.eos_token_id           = _tok.sep_token_id

print("✅ Loaded:", FINAL_MODEL_DIR)
print("Device   :", next(eval_model.parameters()).device)

In [ ]:
@torch.inference_mode()
def predict_images(image_paths, model, processor, device, batch_size=16,
                   num_beams=2, max_new_tokens=24):
    """Batched OCR prediction. Returns one string per image path, same order."""
    model.eval()
    preds = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Predicting"):
        paths  = image_paths[i:i + batch_size]
        images = [Image.open(p).convert("RGB") for p in paths]
        pixel_values = processor(images=images, return_tensors="pt").pixel_values.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
            ids = model.generate(pixel_values, max_new_tokens=max_new_tokens, num_beams=num_beams)
        preds.extend(processor.batch_decode(ids, skip_special_tokens=True))
    return [p.strip() for p in preds]


def evaluate_frame(df, model, processor, device, batch_size=16):
    """Predict every row of df and return (results_df, metrics_dict)."""
    refs  = df["Text"].astype(str).str.strip().tolist()
    preds = predict_images(df["image_path"].tolist(), model, processor, device, batch_size)

    res = pd.DataFrame({
        "Images":     df["Images"].values,
        "image_path": df["image_path"].values,
        "actual":     refs,
        "predicted":  preds,
    })
    res["correct"] = res["actual"].str.lower() == res["predicted"].str.lower()

    metrics = {
        "samples":             int(len(res)),
        "CER":                 float(jiwer.cer(refs, preds)),
        "WER":                 float(jiwer.wer(refs, preds)),
        "exact_match_percent": float(res["correct"].mean() * 100),
    }
    return res, metrics


def save_eval(res, metrics, tag):
    res.to_csv(os.path.join(EVAL_DIR, f"{tag}_predictions.csv"), index=False)
    res[~res["correct"]].to_csv(os.path.join(EVAL_DIR, f"{tag}_incorrect.csv"), index=False)
    metrics = {"model": FINAL_MODEL_NAME, **metrics}
    json.dump(metrics, open(os.path.join(EVAL_DIR, f"{tag}_metrics.json"), "w"), indent=2)
    print(f"✅ Saved {tag} results to {EVAL_DIR}")


def show_metrics(title, m):
    print("=" * 60); print(title); print("=" * 60)
    print(f"Samples            : {m['samples']}")
    print(f"CER                : {m['CER']:.4f}")
    print(f"WER                : {m['WER']:.4f}")
    print(f"Exact-match acc.   : {m['exact_match_percent']:.2f}%")

In [ ]:
# Validation set
val_results, val_metrics = evaluate_frame(val_data, eval_model, eval_processor, device)
show_metrics("VALIDATION RESULTS", val_metrics)
save_eval(val_results, val_metrics, "validation")

In [ ]:
# Held-out TEST set (never used for training or model selection)
test_results, test_metrics = evaluate_frame(test_data, eval_model, eval_processor, device)
show_metrics("TEST RESULTS", test_metrics)
save_eval(test_results, test_metrics, "test")

In [ ]:
# Show a handful of random predictions with their images (not thousands)
N_EXAMPLES = 10
sample_rows = val_results.sample(min(N_EXAMPLES, len(val_results)), random_state=SEED)

for _, row in sample_rows.iterrows():
    plt.figure(figsize=(12, 5))
    plt.imshow(Image.open(row["image_path"]).convert("RGB"))
    plt.axis("off")
    status = "✅ CORRECT" if row["correct"] else "❌ INCORRECT"
    plt.title(f"{status}\nActual    : {row['actual']}\nPredicted : {row['predicted']}", fontsize=11, loc="left")
    plt.show()

## 5. Export — convert `safetensors` to `.pth` for GUI usage

(The original notebook had this heading but never actually produced a `.pth` file.)

In [ ]:
# Inspect saved checkpoints (replaces ~10 separate "find/list" cells)
def describe_checkpoint(folder):
    names = os.listdir(folder)
    weights = [n for n in names if n in ("model.safetensors", "pytorch_model.bin")]
    size = sum(os.path.getsize(os.path.join(folder, n)) for n in weights) / 1024**2
    return weights, size

print(f"{'checkpoint':28s} {'weights':20s} {'size MB':>10s}")
print("-" * 62)
for name in sorted(os.listdir(CHECKPOINT_DIR)):
    folder = os.path.join(CHECKPOINT_DIR, name)
    if os.path.isdir(folder) and (name.startswith("trocr_epoch") or name == "best_trocr_model"):
        w, s = describe_checkpoint(folder)
        print(f"{name:28s} {(w[0] if w else '❌ MISSING'):20s} {s:10.1f}")

In [ ]:
# Verify the checkpoint we are about to export is complete
EXPORT_DIR = os.path.join(CHECKPOINT_DIR, FINAL_MODEL_NAME)

# If the chosen checkpoint does not exist, fall back to the newest saved epoch
if not os.path.isdir(EXPORT_DIR):
    epochs = sorted(
        [d for d in (os.listdir(CHECKPOINT_DIR) if os.path.isdir(CHECKPOINT_DIR) else [])
         if d.startswith("trocr_epoch") and d[len("trocr_epoch"):].isdigit()],
        key=lambda s: int(s[len("trocr_epoch"):]))
    if not epochs:
        raise FileNotFoundError(f"No saved model found in {CHECKPOINT_DIR}")
    print(f"⚠️ '{FINAL_MODEL_NAME}' not found - using '{epochs[-1]}' instead")
    FINAL_MODEL_NAME = epochs[-1]
    EXPORT_DIR = os.path.join(CHECKPOINT_DIR, FINAL_MODEL_NAME)

required = ["config.json"]                       # must exist
optional = ["generation_config.json", "preprocessor_config.json", "tokenizer_config.json",
            "special_tokens_map.json"]           # depends on transformers version
weights, _ = describe_checkpoint(EXPORT_DIR)

missing = [f for f in required if not os.path.exists(os.path.join(EXPORT_DIR, f))]
if not weights:
    missing.append("model weights (model.safetensors / pytorch_model.bin)")

for f in required + optional:
    print("✅" if os.path.exists(os.path.join(EXPORT_DIR, f)) else ("❌" if f in required else "➖ (optional)"), f)
print("✅" if weights else "❌", "weights:", weights)

if missing:
    raise RuntimeError(f"Checkpoint incomplete, missing: {missing}")
print("\n✅ Checkpoint is complete:", EXPORT_DIR)

In [ ]:
# Convert  model.safetensors  ->  .pth   (+ processor files + a loader script for the GUI)
BUNDLE_DIR = "/content/trocr_gui_export"
shutil.rmtree(BUNDLE_DIR, ignore_errors=True)
os.makedirs(BUNDLE_DIR)

export_model = VisionEncoderDecoderModel.from_pretrained(EXPORT_DIR, local_files_only=True)
export_model.eval()

# state_dict() (not safetensors.load_file) so tied weights, e.g. the decoder output head, are included
state_dict = {k: v.detach().cpu() for k, v in export_model.state_dict().items()}

PTH_PATH = os.path.join(BUNDLE_DIR, f"{FINAL_MODEL_NAME}.pth")
torch.save({
    "model_state_dict": state_dict,
    "config": export_model.config.to_dict(),      # needed to rebuild the architecture without the HF folder
    "base_model_name": MODEL_NAME,
    "source_checkpoint": FINAL_MODEL_NAME,
}, PTH_PATH)

# The processor (image preprocessing + tokenizer) is needed by the GUI too
TrOCRProcessor.from_pretrained(EXPORT_DIR, local_files_only=True).save_pretrained(
    os.path.join(BUNDLE_DIR, "processor"))

# Ready-to-use loader for the GUI
with open(os.path.join(BUNDLE_DIR, "load_trocr_pth.py"), "w") as f:
    f.write(f'''import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderConfig, VisionEncoderDecoderModel

PTH_FILE      = "{FINAL_MODEL_NAME}.pth"
PROCESSOR_DIR = "processor"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(PTH_FILE, map_location="cpu")

model = VisionEncoderDecoderModel(VisionEncoderDecoderConfig.from_dict(ckpt["config"]))
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()
processor = TrOCRProcessor.from_pretrained(PROCESSOR_DIR)

# generation settings (needed so decoding stops at end-of-sequence)
model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.generation_config.pad_token_id           = processor.tokenizer.pad_token_id
model.generation_config.eos_token_id           = processor.tokenizer.sep_token_id

def read_prescription(image_path):
    image = Image.open(image_path).convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
    with torch.inference_mode():
        ids = model.generate(pixel_values, max_new_tokens=24, num_beams=4)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
''')

print("✅ .pth created:", PTH_PATH, f"({os.path.getsize(PTH_PATH)/1024**2:.1f} MB)")

# --- Verify: rebuild from the .pth alone and compare every tensor ---
ckpt = torch.load(PTH_PATH, map_location="cpu")
rebuilt = VisionEncoderDecoderModel(VisionEncoderDecoderConfig.from_dict(ckpt["config"]))
result = rebuilt.load_state_dict(ckpt["model_state_dict"], strict=True)
same = all(torch.equal(p, export_model.state_dict()[k].cpu()) for k, p in rebuilt.state_dict().items())
print("load_state_dict:", result)
print("✅ .pth reproduces the original weights exactly" if same else "❌ Weights differ!")

In [ ]:
# Zip the GUI bundle (.pth + processor + loader) and download it.
# For files this large, copying from Drive (below) is usually more reliable than files.download().
from google.colab import files

BUNDLE_ZIP = "/content/trocr_gui_bundle.zip"
with zipfile.ZipFile(BUNDLE_ZIP, "w", zipfile.ZIP_STORED) as z:     # weights don't compress, so STORED is much faster
    for root, _, fs in os.walk(BUNDLE_DIR):
        for f in fs:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, BUNDLE_DIR))

print(f"ZIP size: {os.path.getsize(BUNDLE_ZIP) / 1024**3:.2f} GB")

# Backup copy on Drive (survives runtime disconnects)
drive_copy = os.path.join(CHECKPOINT_DIR, "trocr_gui_bundle.zip")
shutil.copy(BUNDLE_ZIP, drive_copy)
print("Saved to Drive:", drive_copy)

files.download(BUNDLE_ZIP)

In [ ]:
# OPTIONAL: zip the full HuggingFace checkpoint (without the huge optimizer.pt, which is only
# needed for resuming training, not for inference)
HF_ZIP = f"/content/{FINAL_MODEL_NAME}_hf.zip"
with zipfile.ZipFile(HF_ZIP, "w", zipfile.ZIP_STORED) as z:
    for root, _, fs in os.walk(EXPORT_DIR):
        for f in fs:
            if f == "optimizer.pt":
                continue
            full = os.path.join(root, f)
            z.write(full, os.path.join(FINAL_MODEL_NAME, os.path.relpath(full, EXPORT_DIR)))

print("✅", HF_ZIP, f"({os.path.getsize(HF_ZIP) / 1024**3:.2f} GB)")
# from google.colab import files; files.download(HF_ZIP)   # uncomment to download